# 02 Silver: scoped and cleaned data

Builds the analysis dataset from the bronze views created in `01_bronze.ipynb`
(the views live in the shared DuckDB database, so bronze does not need to be re-run).

The thesis scope is applied first, so all later cleaning runs only on the data that is analysed.

## Setup

In [1]:
from config import PLATFORMS, SILVER_ROOT, connect

import pandas as pd

con = connect()
pd.set_option("display.max_rows", 100)

## 1. Apply thesis scope

Silver keeps only statements that fall into the thesis scope:

| Rule | Column | Reason |
|---|---|---|
| Germany is in the territorial scope | `territorial_scope` contains `"DE"` | Thesis focuses on moderation decisions affecting Germany |
| Content posted in 2025 | `year(content_date) = 2025` | The thesis studies moderation of content posted in 2025 |

Note on `territorial_scope`: X always names exactly one country, while TikTok almost always lists the whole EEA.
The DE rule therefore keeps Germany-only statements on X but mostly EEA-wide statements on TikTok. The column
`n_countries_in_scope` is added so that analysis can distinguish Germany-only (`1`) from multi-country statements.

Limitation: the raw data contains only statements **submitted** to the database in 2025 (`created_at`). Content posted
in 2025 but moderated or reported in 2026 is not included, so the last weeks of 2025 are likely undercounted.

The scoped data is defined as views; it is written to parquet once, after the cleaning steps.

In [2]:
con.execute("CREATE SCHEMA IF NOT EXISTS silver")

SCOPE_FILTER = """
    list_contains(from_json(territorial_scope, '["VARCHAR"]'), 'DE')
    AND year(content_date) = 2025
"""

for p in PLATFORMS:
    con.execute(f"""
        CREATE OR REPLACE VIEW silver.scoped_{p} AS
        SELECT *, json_array_length(territorial_scope) AS n_countries_in_scope
        FROM bronze.{p}
        WHERE {SCOPE_FILTER}
    """)

con.sql("SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema = 'silver'").df()

,table_schema,table_name
0,silver,scoped_tiktok
1,silver,scoped_x


Rows remaining after each rule (applied in order). The raw data is grouped by the two scope columns first,
so each platform needs only one scan.

In [3]:
funnel = []
for p in PLATFORMS:
    has_de, in_2025, total = con.sql(f"""
        WITH g AS (
            SELECT territorial_scope, year(content_date) = 2025 AS in_2025, count(*) AS n
            FROM bronze.{p} GROUP BY ALL
        ), f AS (
            SELECT coalesce(list_contains(from_json(territorial_scope, '["VARCHAR"]'), 'DE'), false) AS has_de, in_2025, n
            FROM g
        )
        SELECT sum(n) FILTER (WHERE has_de),
               sum(n) FILTER (WHERE has_de AND coalesce(in_2025, false)),
               sum(n)
        FROM f
    """).fetchone()
    for step, n in [("bronze (all rows)", total), ("territorial_scope contains DE", has_de), ("content_date in 2025", in_2025)]:
        funnel.append({"platform": p, "step": step, "rows": int(n or 0)})

funnel = pd.DataFrame(funnel)
funnel["removed"] = -funnel.groupby("platform")["rows"].diff().fillna(0).astype("int64")
funnel["kept_of_bronze_%"] = (100 * funnel["rows"] / funnel.groupby("platform")["rows"].transform("first")).round(2)
funnel

,platform,step,rows,removed,kept_of_bronze_%
0,tiktok,bronze (all rows),1002729268,0,100.00
1,tiktok,territorial_scope contains DE,999079277,3649991,99.64
2,tiktok,content_date in 2025,979896444,19182833,97.72
3,x,bronze (all rows),670093,0,100.00
4,x,territorial_scope contains DE,183324,486769,27.36
5,x,content_date in 2025,183321,3,27.36
